In [1]:
import networkx as nx 
import pickle 
from dataXplorer import JupyterClient
from dataXplorer import AnalysisToolkit, SessionManager, TextureGenerator, ProjectFileManager, VisualizerSyncer

### script specific functions

In [2]:
import numpy as np  
import random 

def generate_peripheral_position():
    coords = []
    for _ in range(3):
        edge_val = np.random.choice([0.0, 1.0])
        jitter = np.random.uniform(-0.1, 0.1)
        coords.append(np.clip(edge_val + jitter, 0.0, 1.0))
    return tuple(coords)
    
def generate_random_central_position():
    coords = []
    for _ in range(3):
        edge_val = np.random.choice([0.49, 0.51])
        jitter = np.random.uniform(-0.01, 0.01)
        coords.append(np.clip(edge_val + jitter, 0.49,0.51))
    return tuple(coords)

def generate_random_spherical_position(spatial_range=(0.8, 1.0)):
        radius = random.uniform(*spatial_range)
        theta = random.uniform(0, 2 * np.pi)
        phi = random.uniform(0, np.pi)
        x = radius * np.sin(phi) * np.cos(theta)
        y = radius * np.sin(phi) * np.sin(theta)
        z = radius * np.cos(phi)
        
        # Normalize to 0-1 range
        x = (x + 1) / 2
        y = (y + 1) / 2
        z = (z + 1) / 2
        
        return (x, y, z)

import numpy as np

def rotate_positions_around_z(pos_dict, angle_degrees):
    """
    Rotate all 3D positions around Z-axis by given angle (in degrees).
    """
    angle_rad = np.deg2rad(angle_degrees)
    cos_a = np.cos(angle_rad)
    sin_a = np.sin(angle_rad)

    rotation_matrix = np.array([
        [cos_a, -sin_a, 0],
        [sin_a,  cos_a, 0],
        [0,      0,     1]
    ])

    rotated_pos = {}
    for key, (x, y, z) in pos_dict.items():
        vec = np.array([x, y, z])
        new_vec = rotation_matrix @ vec
        rotated_pos[key] = tuple(new_vec)

    return rotated_pos





### Connect

In [3]:
# run backend (server) using buildandrun powershell script
client = JupyterClient()

✅ Connected to /main


✅ Connected to /main
✅ Connected to /main


In [4]:
#client.disconnect()

# Prerequisites & Infos before starting: 

In [5]:
# FIRST : upload a Graph with all needed information as node annotation attributes
# (e.g. this can include initial positions in case you want to access them and 
# for time reasons start off with a precalculated layout, node types if there are 
# several graphs merged into one, ... )

# then use the function "make_json(G)" in nx2json and upload the graph.
# this step could be integrated into the notebook though. 
 
# then you can run the cells in this notebook one by one. 

# Note: input files are stored locally and not shared via git (temp-files folder)

# Get started

In [6]:
# see if new project is in projectlist
import GlobalData as GD

allprojects_updated = []
for i, proj in enumerate(GD.listProjects()):
    allprojects_updated.append((i,proj))
allprojects_updated

[(0, 'CDK5'),
 (1, 'CircLadderGraph-xsmall'),
 (2, 'diffusion'),
 (3, 'Exposurome'),
 (4, 'GenExpression_01'),
 (5, 'GenExpression_02'),
 (6, 'imunet_250130-X0-inter'),
 (7, 'imunet_250130-X1-inter'),
 (8, 'JSON_autocore'),
 (9, 'JSON_barbellgraph'),
 (10, 'JSON_Zachary'),
 (11, 'Microplastics_HumanHealth'),
 (12, 'Pesticides_HumanHealth'),
 (13, 'Powergrid_Europe'),
 (14, 'PPI_brain_infarction'),
 (15, 'Sphere_Torus'),
 (16, 'Sphere_Torus_Morph'),
 (17, 'Teapot'),
 (18, 'TEAPOT-RealtimeTesting'),
 (19, 'TheMandelbulb_edges')]

In [7]:
# select a project to work with
sel_id = 12
sel_name = allprojects_updated[sel_id][1]

# load the project data
session = SessionManager(sel_id, sel_name, client)
session.load_graph_from_project()

session.reload_project()

# initialize 
file_mgr = ProjectFileManager(session)
tex_gen = TextureGenerator(session)
syncer = VisualizerSyncer(session)
tools = AnalysisToolkit(session, tex_gen, syncer, file_mgr)

Session Graph loaded from project folder. Data: Nodes: 15348 Links: 10467


In [8]:
# add waiting time for the server to be ready
import time
time.sleep(3)

latest_message = client.latest_data
session.get_active_layouts(latest_message)

Layout: scene01-Cities_geo Selected index: 0
Layout: scene01-Cities_geo Selected index: 0
Layout: scene01-Cities_geo Selected index: 0


### Reconstruct Graph from Project

In [9]:
G = session.load_graph_from_project()
print("G_nodes:", len(G.nodes()))
print("G_edges:", len(G.edges()))

Session Graph loaded from project folder. Data: Nodes: 15348 Links: 10467
G_nodes: 15348
G_edges: 10467


In [10]:
# check for node attributes 
node_attr = G.nodes(data=True)

# quick check
node_attr[1]

{'name': 'Albania',
 'attrlist': {'original_id': 1,
  'new_id': 1,
  'name': 'Albania',
  'init_pos': [0.6301777113349858, 0.8544170422477306, 0.8277839587837629],
  'type': 'Europe',
  'nodecolor': [194, 78, 2, 109]}}

In [11]:
# get graph attributes 
init_nodecolors = {}
for node in G.nodes():
    init_nodecolors[node] = node_attr[node]["attrlist"]["nodecolor"]
nx.set_node_attributes(G, init_nodecolors, 'nodecolor')

node_type = {}
for node in G.nodes():
    node_type[node] = node_attr[node]["attrlist"]["type"]

In [12]:
node_type

{0: 'Asia',
 1: 'Europe',
 2: 'Africa',
 3: 'Europe',
 4: 'Africa',
 5: 'North America',
 6: 'South America',
 7: 'Europe',
 8: 'Oceania',
 9: 'Europe',
 10: 'Europe',
 11: 'North America',
 12: 'Asia',
 13: 'Asia',
 14: 'North America',
 15: 'Europe',
 16: 'Europe',
 17: 'North America',
 18: 'Africa',
 19: 'Asia',
 20: 'South America',
 21: 'Europe',
 22: 'Africa',
 23: 'South America',
 24: 'Asia',
 25: 'Europe',
 26: 'Africa',
 27: 'Africa',
 28: 'Africa',
 29: 'Africa',
 30: 'Asia',
 31: 'Africa',
 32: 'North America',
 33: 'Africa',
 34: 'Africa',
 35: 'South America',
 36: 'Asia',
 37: 'South America',
 38: 'Africa',
 39: 'Africa',
 40: 'North America',
 41: 'Europe',
 42: 'North America',
 43: 'Europe',
 44: 'Europe',
 45: 'Africa',
 46: 'Europe',
 47: 'Africa',
 48: 'North America',
 49: 'North America',
 50: 'South America',
 51: 'Africa',
 52: 'North America',
 53: 'Africa',
 54: 'Africa',
 55: 'Europe',
 56: 'Africa',
 57: 'Africa',
 58: 'Oceania',
 59: 'Europe',
 60: 'Euro

### preprocessing of latitude, longitude 


In [13]:
import pandas as pd

from math import radians, cos, sin, sqrt, atan, atan2
def geodetic_to_geocentric(ellipsoid, lat, lon, height):
    # Adjust the flattening factor to influence the ellipsoid shape
    a, rf = ellipsoid
    # rf *= 1.2  # Increase the flattening factor to bring poles closer together

    phi = radians(lat)
    lamb = radians(lon)

    sin_phi = sin(phi)
    e2 = 1 - (1 - 1 / rf) ** 2
    n = a / sqrt(1 - e2 * sin_phi ** 2) 
    
    r = ((n + height) * cos(phi))
    x = r * cos(lamb)
    y = r * sin(lamb)   
    z = (n * (1 - e2) + height) * sin(phi) 

    return x, y, z  


def normalize_coordinates(x, y, z):
    magnitude = sqrt(x**2 + y**2 + z**2)
    x_norm, y_norm, z_norm = x / magnitude, y / magnitude, z / magnitude
    x_final = (x_norm+1)/2
    y_final = (y_norm+1)/2
    z_final = (z_norm+1)/2
    return x_final, y_final, z_final


df = pd.read_csv("temp-files/Pesticides/world-data-2023.csv")
pos_latlon = df[["Latitude", "Longitude"]].values.tolist()
d_pos_latlon = dict(zip(df["Country"], pos_latlon))

# modify to rotate 90 degrees into the other direction around z-axis
d_pos_latlon_rotated = {}
for k, v in d_pos_latlon.items():
    lat, lon = v
    # Rotate the longitude by -90 degrees
    lon_rotated = (lon - 90) % 360 #- 90
    d_pos_latlon_rotated[k] = (lat, lon_rotated)

pos_xyz = {}

flattening_factor = 298.257223563
WGS84_radius = 6378137.0

for k,v in d_pos_latlon_rotated.items():
    x,y,z = geodetic_to_geocentric((WGS84_radius, flattening_factor), v[0],v[1], 0)
    xn, yn, zn = normalize_coordinates(x,y,z)
    pos_xyz[k] = (yn,xn,zn)

# fix NaN values in init_pos
for i, k in pos_xyz.items():
    if np.isnan(k[0]) or np.isnan(k[1]) or np.isnan(k[2]):
        pos_xyz[i] = generate_random_spherical_position(spatial_range=(0.8, 1.0))

In [14]:
df

,Country,Density\n(P/Km2),Abbreviation,Agricultural Land( %),Land Area(Km2),Armed Forces size,Birth Rate,Calling Code,Capital/Major City,Co2-Emissions,...,Out of pocket health expenditure,Physicians per thousand,Population,Population: Labor force participation (%),Tax revenue (%),Total tax rate,Unemployment rate,Urban_population,Latitude,Longitude
0,Afghanistan,60,AF,58.10%,"652,230","323,000",32.49,93.0,Kabul,"8,672",...,78.40%,0.28,"38,041,754",48.90%,9.30%,71.40%,11.12%,"9,797,273",33.939110,67.709953
1,Albania,105,AL,43.10%,"28,748","9,000",11.78,355.0,Tirana,"4,536",...,56.90%,1.20,"2,854,191",55.70%,18.60%,36.60%,12.33%,"1,747,593",41.153332,20.168331
2,Algeria,18,DZ,17.40%,"2,381,741","317,000",24.28,213.0,Algiers,"150,006",...,28.10%,1.72,"43,053,054",41.20%,37.20%,66.10%,11.70%,"31,510,100",28.033886,1.659626
3,Andorra,164,AD,40.00%,468,NaN,7.20,376.0,Andorra la Vella,469,...,36.40%,3.33,"77,142",NaN,NaN,NaN,NaN,"67,873",42.506285,1.521801
4,Angola,26,AO,47.50%,"1,246,700","117,000",40.73,244.0,Luanda,"34,693",...,33.40%,0.21,"31,825,295",77.50%,9.20%,49.10%,6.89%,"21,061,025",-11.202692,17.873887
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,Venezuela,32,VE,24.50%,"912,050","343,000",17.88,58.0,Caracas,"164,175",...,45.80%,1.92,"28,515,829",59.70%,NaN,73.30%,8.80%,"25,162,368",6.423750,-66.589730
191,Vietnam,314,VN,39.30%,"331,210","522,000",16.75,84.0,Hanoi,"192,668",...,43.50%,0.82,"96,462,106",77.40%,19.10%,37.60%,2.01%,"35,332,140",14.058324,108.277199
192,Yemen,56,YE,44.60%,"527,968","40,000",30.45,967.0,Sanaa,"10,609",...,81.00%,0.31,"29,161,922",38.00%,NaN,26.60%,12.91%,"10,869,523",15.552727,48.516388
193,Zambia,25,ZM,32.10%,"752,618","16,000",36.19,260.0,Lusaka,"5,141",...,27.50%,1.19,"17,861,030",74.60%,16.20%,15.60%,11.43%,"7,871,713",-13.133897,27.849332


In [15]:
agriculture_percent = df["Agricultural Land( %)"].values.tolist()
# remove % from values
agriculture_percent = [float(str(x).replace("%", "").replace(",", ".")) for x in agriculture_percent]
# multiply to get int instead of percentage 
agriculture_percent = [x * 100 for x in agriculture_percent]

# if nan values set them 0
for i, k in enumerate(agriculture_percent):
    if np.isnan(k):
        agriculture_percent[i] = 0

agriculture_percent = dict(zip(df["Country"], agriculture_percent))
agriculture_percent

{'Afghanistan': 5810.0,
 'Albania': 4310.0,
 'Algeria': 1739.9999999999998,
 'Andorra': 4000.0,
 'Angola': 4750.0,
 'Antigua and Barbuda': 2050.0,
 'Argentina': 5430.0,
 'Armenia': 5890.0,
 'Australia': 4820.0,
 'Austria': 3240.0,
 'Azerbaijan': 5770.0,
 'The Bahamas': 140.0,
 'Bahrain': 1110.0,
 'Bangladesh': 7059.999999999999,
 'Barbados': 2330.0,
 'Belarus': 4200.0,
 'Belgium': 4460.0,
 'Belize': 700.0,
 'Benin': 3329.9999999999995,
 'Bhutan': 1360.0,
 'Bolivia': 3479.9999999999995,
 'Bosnia and Herzegovina': 4310.0,
 'Botswana': 4560.0,
 'Brazil': 3390.0,
 'Brunei': 270.0,
 'Bulgaria': 4630.0,
 'Burkina Faso': 4420.0,
 'Burundi': 7920.0,
 'Ivory Coast': 6480.0,
 'Cape Verde': 1960.0000000000002,
 'Cambodia': 3090.0,
 'Cameroon': 2060.0,
 'Canada': 690.0,
 'Central African Republic': 819.9999999999999,
 'Chad': 3970.0000000000005,
 'Chile': 2120.0,
 'China': 5620.0,
 'Colombia': 4029.9999999999995,
 'Comoros': 7150.0,
 'Republic of the Congo': 3110.0,
 'Costa Rica': 3450.0,
 'Croati

# SCENES

### SCENE 1 - agriculture land use + globe 

In [27]:
endrin_pos = {}
for node in G.nodes():
    if node_type[node] == 'endrin':
        endrin_pos[node] = G.nodes[node]["attrlist"]["init_pos"]

city_nodes = []
for node, tpe in node_type.items():
    if tpe != 'ppi' and tpe != 'endrin':
        city_nodes.append(node)

# add random positions to the rest of the nodes from endrin_pos
for i in G.nodes():
    if i in city_nodes: #node_type[i] == "city":
        continue
    else:
        pos_xyz[i] = random.choice(list(endrin_pos.values()))

In [28]:
pos_xyz # correct rotation/positions relative to globe if first pos:  0.34, 0.88, 0.77

{'Afghanistan': (0.3423340592772383, 0.8846195825311766, 0.7778654852074114),
 'Albania': (0.14558295775226937, 0.6301777113349855, 0.8277839587837628),
 'Algeria': (0.05819863445128348, 0.5128007712440266, 0.7337684616342579),
 'Andorra': (0.1304008907531628, 0.5098190233041598, 0.8366007801911588),
 'Angola': (0.03308236738555265, 0.6505753192162869, 0.4034857647979675),
 'Antigua and Barbuda': (0.2739660701734179,
  0.07851160027993997,
  0.6457950323864798),
 'Argentina': (0.3254618794832421, 0.14813819520550253, 0.19059781058751918),
 'Armenia': (0.22886643861797035, 0.7714952364374124, 0.820588409778162),
 'Australia': (0.8131828269119072, 0.8268678574571959, 0.2876938044031484),
 'Austria': (0.1719470525205417, 0.5851461973538405, 0.8676022153445495),
 'Azerbaijan': (0.241437627350931, 0.7829330291959546, 0.8210831674817057),
 'The Bahamas': (0.40102811877773886,
  0.057360252579655435,
  0.7104153528879966),
 'Bahrain': (0.21429168813618166, 0.8473039070705919, 0.71851946518657

In [29]:
id_range_cities = list(range(len(G.nodes())))
pos_xyz_ids = dict(zip(id_range_cities, pos_xyz.values()))

In [30]:
nx.set_node_attributes(G, pos_xyz_ids, 'pos')

In [31]:
# node colors 
nodecol_cities = {}
for i, k in pos_xyz_ids.items():
    if i not in city_nodes:
        nodecol_cities[i] = (0,0,0,0)
    else:
        nodecol_cities[i] = G.nodes[i]['nodecolor']

linkcol_cities = {} # empty : all edges should be black 

In [32]:
# layout_name = "scene01-Cities_geo"
# new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_cities, layout_name, save=True)
# new_tex_links = tex_gen.generate_link_color_texture(linkcol_cities, layout_name, save=True)
# new_tex_nodepos = tex_gen.generate_node_position_texture(pos_xyz, layout_name, save=True, normalize_flag=False)

# nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
# linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
# nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
# nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

# session.client.emit("ex", {
#     "usr": session.client.uid,
#     "id":None,
#     "fn": "updateTempTex",
#     "textures": [
#         {"channel": "nodeRGB", "path": nodeRGB_path_rel},
#         {"channel": "linkRGB", "path": linksRGB_path_rel},
#         {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
#         {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
#     ]
# }, namespace=session.client.namespace)


### SCENE 02 - agriculture land use + globe with areas  

In [33]:
# create areas around city nodes with remaining nodes (non-city nodes) 
# and assign them to the city node

# Assign non-city nodes to city nodes based on alpha values
city_areas = {city: [] for city in city_nodes}
non_city_nodes = [node for node in G.nodes() if node not in city_nodes]

# Calculate the total agriculture percentage
total_agriculture = sum(agriculture_percent.values())

# Calculate the number of non-city nodes to assign to each city node based on agriculture percentage
city_node_allocation = {
    city: int((agriculture_percent[G.nodes[city]["attrlist"]["name"]] / total_agriculture) * len(non_city_nodes))
    for city in city_nodes
}

# Assign non-city nodes to city nodes
non_city_nodes_iter = iter(non_city_nodes)
for city, count in city_node_allocation.items():
    for _ in range(count):
        try:
            node = next(non_city_nodes_iter)
            city_areas[city].append(node)
        except StopIteration:
            break


In [34]:
d_cityareas = dict(zip(d_pos_latlon.keys(), city_areas.values()))
len(d_cityareas)

195

In [35]:
d_cityareas_len = {}
for k, v in d_cityareas.items():
    d_cityareas_len[k] = len(v)
d_cityareas_len

{'Afghanistan': 119,
 'Albania': 88,
 'Algeria': 35,
 'Andorra': 82,
 'Angola': 97,
 'Antigua and Barbuda': 42,
 'Argentina': 111,
 'Armenia': 121,
 'Australia': 99,
 'Austria': 66,
 'Azerbaijan': 118,
 'The Bahamas': 2,
 'Bahrain': 22,
 'Bangladesh': 145,
 'Barbados': 48,
 'Belarus': 86,
 'Belgium': 91,
 'Belize': 14,
 'Benin': 68,
 'Bhutan': 28,
 'Bolivia': 71,
 'Bosnia and Herzegovina': 88,
 'Botswana': 93,
 'Brazil': 69,
 'Brunei': 5,
 'Bulgaria': 95,
 'Burkina Faso': 91,
 'Burundi': 163,
 'Ivory Coast': 133,
 'Cape Verde': 40,
 'Cambodia': 63,
 'Cameroon': 42,
 'Canada': 14,
 'Central African Republic': 16,
 'Chad': 81,
 'Chile': 43,
 'China': 115,
 'Colombia': 83,
 'Comoros': 147,
 'Republic of the Congo': 64,
 'Costa Rica': 71,
 'Croatia': 56,
 'Cuba': 123,
 'Cyprus': 25,
 'Czech Republic': 93,
 'Democratic Republic of the Congo': 23,
 'Denmark': 127,
 'Djibouti': 151,
 'Dominica': 68,
 'Dominican Republic': 100,
 'Ecuador': 45,
 'Egypt': 7,
 'El Salvador': 157,
 'Equatorial Gui

In [36]:
# create positions for the non-city nodes based on the city node positions
pos_latlon_noncitynodes = {}
node_type_noncitynodes = {}

offset_uniform = 1.2
offset_normal = 0.3

for x,(city, nodes) in enumerate(d_cityareas.items()):
    city_lat, city_lon = d_pos_latlon[city]
    for node in nodes:
        # adjust offset based on number of nodes in the area 

        # UNIFORM NOISE
        # Generate random latitude and longitude offsets for each node
        #lat_offset = np.random.uniform(-offset_uniform, offset_uniform) * np.random.uniform(0.2, 1.8) * np.random.choice([0.5, 1.0, 1.5])
        #lon_offset = np.random.uniform(-offset_uniform, offset_uniform) * np.random.uniform(0.2, 1.8) * np.random.choice([0.5, 1.0, 1.5])
        # Introduce some randomness to create irregular blobs
        #lat_offset += np.random.uniform(-0.05, 0.05) * np.sin(np.random.uniform(0, 2 * np.pi))
        #lon_offset += np.random.uniform(-0.05, 0.05) * np.cos(np.random.uniform(0, 2 * np.pi))

        # GAUSSIAN NOISE + UNIFORM RANDOM BLOB
        lat_offset = np.random.normal(-offset_normal, offset_normal) * np.random.normal(0.2, 1.4) * np.random.choice([0.5, 1.0, 1.5])
        lon_offset = np.random.normal(-offset_normal, offset_normal) * np.random.normal(0.2, 1.4) * np.random.choice([0.5, 1.0, 1.5])
        # Introduce some randomness to create irregular blobs
        lat_offset += np.random.uniform(-0.05, 0.05) * np.sin(np.random.uniform(0, 2 * np.pi))
        lon_offset += np.random.uniform(-0.05, 0.05) * np.cos(np.random.uniform(0, 2 * np.pi))


        new_lat = np.clip(city_lat + lat_offset, -90, 90)
        new_lon = (city_lon + lon_offset + 180) % 360 - 180  # Ensure longitude is within [-180, 180]
        
        pos_latlon_noncitynodes[node] = (new_lat, new_lon)
        node_type_noncitynodes[node] = G.nodes[x]["attrlist"]["type"]


In [37]:
node_type_city_nodes = {}
for i in city_nodes:
    if i in node_type.keys():
        node_type_city_nodes[i] = node_type[i]

len(node_type_city_nodes)

195

In [38]:
node_type_scene3 = {**node_type_city_nodes, **node_type_noncitynodes}

In [39]:
for g in G.nodes():
    if g not in node_type_scene3.keys():
        node_type_scene3[g] = G.nodes[g]["attrlist"]["type"]
        print("added node:", g, G.nodes[g]["attrlist"]["type"])
len(node_type_scene3)


added node: 15254 ppi
added node: 15255 ppi
added node: 15256 ppi
added node: 15257 ppi
added node: 15258 ppi
added node: 15259 ppi
added node: 15260 ppi
added node: 15261 ppi
added node: 15262 ppi
added node: 15263 ppi
added node: 15264 ppi
added node: 15265 ppi
added node: 15266 ppi
added node: 15267 ppi
added node: 15268 ppi
added node: 15269 ppi
added node: 15270 ppi
added node: 15271 ppi
added node: 15272 ppi
added node: 15273 ppi
added node: 15274 ppi
added node: 15275 ppi
added node: 15276 ppi
added node: 15277 ppi
added node: 15278 ppi
added node: 15279 ppi
added node: 15280 ppi
added node: 15281 ppi
added node: 15282 ppi
added node: 15283 ppi
added node: 15284 ppi
added node: 15285 ppi
added node: 15286 ppi
added node: 15287 ppi
added node: 15288 ppi
added node: 15289 ppi
added node: 15290 ppi
added node: 15291 ppi
added node: 15292 ppi
added node: 15293 ppi
added node: 15294 ppi
added node: 15295 ppi
added node: 15296 ppi
added node: 15297 ppi
added node: 15298 ppi
added node

15348

In [40]:
len(G.nodes())

15348

In [41]:
# check for nan values
for i, k in pos_latlon_noncitynodes.items():
    if np.isnan(k[0]) or np.isnan(k[1]):
        print(i,k)
        pos_latlon_noncitynodes[i] = (0,0)

11373 (nan, nan)
11374 (nan, nan)
11375 (nan, nan)
11376 (nan, nan)
11377 (nan, nan)
11378 (nan, nan)
11379 (nan, nan)
11380 (nan, nan)
11381 (nan, nan)
11382 (nan, nan)
11383 (nan, nan)
11384 (nan, nan)
11385 (nan, nan)
11386 (nan, nan)
11387 (nan, nan)
11388 (nan, nan)
11389 (nan, nan)
11390 (nan, nan)
11391 (nan, nan)
11392 (nan, nan)
11393 (nan, nan)
11394 (nan, nan)
11395 (nan, nan)
11396 (nan, nan)
11397 (nan, nan)
11398 (nan, nan)
11399 (nan, nan)
11400 (nan, nan)
11401 (nan, nan)
11402 (nan, nan)
11403 (nan, nan)
11404 (nan, nan)
11405 (nan, nan)
11406 (nan, nan)
11407 (nan, nan)
11408 (nan, nan)
11409 (nan, nan)
11410 (nan, nan)
11411 (nan, nan)
11412 (nan, nan)
11413 (nan, nan)
11414 (nan, nan)
11415 (nan, nan)
11416 (nan, nan)
11417 (nan, nan)
11418 (nan, nan)
11419 (nan, nan)
11420 (nan, nan)
11421 (nan, nan)
11422 (nan, nan)
11423 (nan, nan)
11424 (nan, nan)
11425 (nan, nan)
11426 (nan, nan)
11427 (nan, nan)
11428 (nan, nan)
11429 (nan, nan)
11430 (nan, nan)
11431 (nan, na

In [42]:
# convert to xyz coordinates

pos_latlon_noncitynodes_rotated = {}
for k, v in pos_latlon_noncitynodes.items():
    lat, lon = v
    # Rotate the longitude by -90 degrees
    lon_rotated = (lon - 90) % 360 #- 90
    pos_latlon_noncitynodes_rotated[k] = (lat, lon_rotated)

pos_xyz_noncitynodes = {}
for k,v in pos_latlon_noncitynodes_rotated.items():
    x,y,z = geodetic_to_geocentric((WGS84_radius, flattening_factor), v[0],v[1], 0)
    xn, yn, zn = normalize_coordinates(x,y,z)
    pos_xyz_noncitynodes[k] = (yn,xn,zn)

In [43]:
# merge both city coordainates and non-city coordinates into one dict
pos_areas = {**pos_xyz_ids, **pos_xyz_noncitynodes}

In [44]:
# check if same as pos_areas
len(G.nodes()) == len(pos_areas)

True

In [45]:
nodecol_areas = {}
for i, k in pos_areas.items():
    if i in city_nodes:
        nodecol_areas[i] = G.nodes[i]['nodecolor']
    else:
        # Find the city node this node belongs to
        for city, nodes in city_areas.items():
            if i in nodes:
                nodecol_areas[i] =  G.nodes[city]['nodecolor']#(G.nodes[city]['nodecolor'][0], G.nodes[city]['nodecolor'][1], G.nodes[city]['nodecolor'][2], 80) #G.nodes[city]['nodecolor']
                break

In [46]:
linkcol_areas = {} # empty : all edges should be black 

In [47]:
layout_name = "scene01-Cities_geo"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_cities, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_cities, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_areas, layout_name, save=True, normalize_flag=False)

nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


In [48]:
layout_name = "scene02-Cities-areas_geo"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_areas, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_areas, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_areas, layout_name, save=True, normalize_flag=False)

nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 3 - Endrin with US and Europe

In [49]:
endrin_nodes = []
for node, attr in node_type.items():
    if 'endrin' in attr:
        endrin_nodes.append(node)
print(len(endrin_nodes))

800


In [50]:
# node positions

pos_endrin = {}
nodecol_endrin = {}

col_endrin_rgba = G.nodes[195]["attrlist"]["nodecolor"] #(200,80,0,180) 

for i in G.nodes():
    if i in endrin_nodes:
        pos_endrin[i] = G.nodes[i]["attrlist"]["init_pos"]
        nodecol_endrin[i] = col_endrin_rgba
    elif node_type_scene3[i] == 'North America':
        pos_endrin[i] =  pos_areas[i] #G.nodes[i]['pos']
        nodecol_endrin[i] = (255,0,0,150) #nodecol_areas[i]
    elif node_type_scene3[i] == 'Europe':
        pos_endrin[i] = pos_areas[i] #G.nodes[i]['pos']
        nodecol_endrin[i] = (255,0,0,150) #nodecol_areas[i]
    else:
        pos_endrin[i] = pos_areas[i] # generate_random_spherical_position(spatial_range=(0.0,0.00001))
        nodecol_endrin[i] = (20,20,20,20) # (0,0,0,0)


In [51]:
# link colors

linkcol_endrin = {}
l_endrin_edges = []

for i in G.edges():
    if i[0] in endrin_nodes and i[1] in endrin_nodes:
        l_endrin_edges.append(i)
        linkcol_endrin[i] = col_endrin_rgba
    else:
        linkcol_endrin[i] = (0,0,0,0)

In [52]:
layout_name = "scene03-Endrin-molecule"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_endrin, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_endrin, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_endrin, layout_name, save=True, normalize_flag=False)


# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 4 - Endrin with ppi

In [53]:
ppi_nodes = []  
for node, attr in node_type.items():
    if 'ppi' in attr:
        ppi_nodes.append(node)
print(len(ppi_nodes))

14353


In [54]:
endrin_protein_nodes = [] 
with open ("temp-files/Pesticides/Endrin_genes.txt", "r") as f:
    for line in f:
        endrin_protein_nodes.append(line.strip())

disease_genes = []
with open ("temp-files/Pesticides/Endrin_diseases_proximity_dict.pickle", "rb") as f:
    disease_genes = pickle.load(f)

In [55]:
node_name = {}
for node in G.nodes():
    node_name[node] = node_attr[node]["attrlist"]["name"]

endrin_protein_nodes_ids = []
for node in G.nodes():
    if node_name[node] in endrin_protein_nodes:
        endrin_protein_nodes_ids.append(node)

In [56]:
# node positions

pos_endrin_ppi = {}
for i in G.nodes():
    if i in endrin_nodes:
        pos_endrin_ppi[i] = G.nodes[i]["attrlist"]["init_pos"]
    elif i in ppi_nodes:
        pos_endrin_ppi[i] = G.nodes[i]["attrlist"]["init_pos"]
    else:
        pos_endrin_ppi[i] = generate_random_spherical_position(spatial_range=(0.0,0.05))

In [57]:
# node colors

import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib as mpl    

col_palette = cm.get_cmap('Blues', len(ppi_nodes))
norm = mpl.colors.Normalize(vmin=0, vmax=len(ppi_nodes)-1)
cmap = mpl.cm.ScalarMappable(norm=norm, cmap=col_palette)
cmap.set_array([])

d_nodecolors_cmap = {}
for i, node in enumerate(ppi_nodes):
    rgba = cmap.to_rgba(i)
    d_nodecolors_cmap[node] = (rgba[0]*255, rgba[1]*255, rgba[2]*255, 70)

nodecol_endrin_ppi = {}
for i in G.nodes():
    if i in endrin_nodes:
        nodecol_endrin_ppi[i] = col_endrin_rgba
    elif i in endrin_protein_nodes_ids:
        nodecol_endrin_ppi[i] = (255,80,0,200) #(0,0,255,200)
    elif i in ppi_nodes:
        nodecol_endrin_ppi[i] = d_nodecolors_cmap[i]
    else:
        nodecol_endrin_ppi[i] = (0,0,0,0)

C:\Users\chris\AppData\Local\Temp\ipykernel_36264\3387959417.py:7: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  col_palette = cm.get_cmap('Blues', len(ppi_nodes))


In [58]:
# link colors

linkcol_endrin_ppi = {}
l_endrin_edges = []

for i in G.edges():
    if i[0] in endrin_nodes and i[1] in endrin_nodes:
        l_endrin_edges.append(i)
        linkcol_endrin_ppi[i] = col_endrin_rgba
    elif i[0] in ppi_nodes and i[1] in ppi_nodes:
        linkcol_endrin_ppi[i] = (0,122,192,200)
    else:
        linkcol_endrin_ppi[i] = (0,0,0,0)

In [59]:
layout_name = "scene04-Endrin-molecule-ppi"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_endrin_ppi, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_endrin_ppi, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_endrin_ppi, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 5 - Endrin with ppi and 25 proteins

In [60]:
d_endrin_protein_nodes = {}
for node in G.nodes():
    if node_name[node] in endrin_protein_nodes:
        d_endrin_protein_nodes[node] = node_attr[node]["attrlist"]["name"]

In [61]:
d_endrin_protein_nodes

{1501: 'ESR2',
 1637: 'ALK',
 1981: 'NTRK1',
 2041: 'ESR1',
 6233: 'NTRK2',
 6234: 'RET',
 6750: 'PDGFRB',
 7024: 'KIT',
 7098: 'RARB',
 7099: 'RARG',
 7169: 'PGR',
 7310: 'EPHA8',
 8642: 'RORA',
 10540: 'GFRA1',
 11455: 'BHLHA15',
 11590: 'CYP2B6',
 14512: 'LRRC2',
 15226: 'CYP26A1'}

In [62]:
for i in G.nodes():
    if "CALR" == G.nodes[i]['name']:
        print(G.nodes[i]['name'], i)
    elif "BCR" == G.nodes[i]['name']:
        print(G.nodes[i]['name'], i)
    elif "NTRK1" == G.nodes[i]['name']:
        print(G.nodes[i]['name'], i)

NTRK1 1981
CALR 2913
BCR 5637


In [63]:
# positions 

pos_endrin_ppi25 = {}
for i in G.nodes():
    if i in endrin_nodes:
        pos_endrin_ppi25[i] = G.nodes[i]["attrlist"]["init_pos"]
    elif i in endrin_protein_nodes_ids:
        pos_endrin_ppi25[i] = generate_random_spherical_position(spatial_range=(0.3,0.4))
    elif i in ppi_nodes and i not in endrin_protein_nodes_ids:
        pos_endrin_ppi25[i] = G.nodes[i]["attrlist"]["init_pos"]
    else:
        pos_endrin_ppi25[i] = generate_random_spherical_position(spatial_range=(0.0,0.05))

In [64]:
# node colors

nodecol_endrin_ppi25 = {}
for i in G.nodes():
    if i in endrin_nodes:
        nodecol_endrin_ppi25[i] = col_endrin_rgba
    elif i in endrin_protein_nodes_ids:
        nodecol_endrin_ppi25[i] = (255,80,0,200) #(0,0,255,200)
    elif i in ppi_nodes:
        nodecol_endrin_ppi25[i] = (20,20,20,20) #(d_nodecolors_cmap[i][0], d_nodecolors_cmap[i][1], d_nodecolors_cmap[i][2], 10)
    else:
        nodecol_endrin_ppi25[i] = (0,0,0,0)

In [65]:
# link colors

linkcol_endrin_ppi25 = {}

for i in G.edges():
    if i[0] in endrin_nodes and i[1] in endrin_nodes:
        linkcol_endrin_ppi25[i] = col_endrin_rgba
    elif i[0] in endrin_protein_nodes_ids or i[1] in endrin_protein_nodes_ids:
        linkcol_endrin_ppi25[i] = (0,0,0,0)
    elif i[0] in ppi_nodes and i[1] in ppi_nodes:
        linkcol_endrin_ppi25[i] = (20,20,20,20) # (0,122,192,100)
    else:
        linkcol_endrin_ppi25[i] = (0,0,0,0)

In [66]:
layout_name = "scene05-Endrin-molecule-ppi-25proteins"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_endrin_ppi25, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_endrin_ppi25, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_endrin_ppi25, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 6 - hämatologischer Krebs

In [67]:
import pickle
with open ('temp-files/Pesticides/Endrin_diseases_proximity_dict.pickle', 'rb') as f:
    d_disease_genes = pickle.load(f)

for i,n in d_disease_genes.items():
    print(i, len(n))

Colorectal Carcinoma 77
Malignant neoplasm of pancreas 11
Malignant neoplasm of breast 88
Breast Carcinoma 73
ovarian neoplasm 27
Malignant neoplasm of ovary 28
Status Epilepticus 24
Mammary Neoplasms, Experimental 18
Prostatic Neoplasms 14
Malignant neoplasm of prostate 36
Mammary Neoplasms 24
MYELODYSPLASTIC SYNDROME 22
Leukemia, Myelocytic, Acute 53


In [68]:
dismod_1 = list(d_disease_genes["MYELODYSPLASTIC SYNDROME"])+list(d_disease_genes["Leukemia, Myelocytic, Acute"])
print("len dismod_1:", len(dismod_1))   

dismod_2 = list(d_disease_genes["Breast Carcinoma"])
print("len dismod_2:", len(dismod_2))

dismod_3 = list(d_disease_genes["Colorectal Carcinoma"])
print("len dismod_3:", len(dismod_3))


len dismod_1: 75
len dismod_2: 73
len dismod_3: 77


In [69]:
d_id_nodename = {}
for node in G.nodes():
    d_id_nodename[node] = G.nodes[node]['name']

In [70]:
d_id_dismod_1 = {}
d_id_dismod_2 = {}
d_id_dismod_3 = {}

for k,v in d_id_nodename.items():
    if v in dismod_1:
        d_id_dismod_1[k] = v
    elif v in dismod_2:
        d_id_dismod_2[k] = v
    elif v in dismod_3:
        d_id_dismod_3[k] = v

In [71]:
# make subgraphs 

G_dismod_1 = G.subgraph(d_id_dismod_1.keys())
print("G_dismod_1_nodes:", len(G_dismod_1.nodes()))
print("G_dismod_1_edges:", len(G_dismod_1.edges()))

G_dismod_2 = G.subgraph(d_id_dismod_2.keys())
print("G_dismod_2_nodes:", len(G_dismod_2.nodes()))
print("G_dismod_2_edges:", len(G_dismod_2.edges()))

G_dismod_3 = G.subgraph(d_id_dismod_3.keys())
print("G_dismod_3_nodes:", len(G_dismod_3.nodes()))
print("G_dismod_3_edges:", len(G_dismod_3.edges()))

G_dismod_1_nodes: 65
G_dismod_1_edges: 68
G_dismod_2_nodes: 66
G_dismod_2_edges: 172
G_dismod_3_nodes: 65
G_dismod_3_edges: 65


In [74]:
# node positions 

pos_diseases_1 = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
        pos_dis_1 = nx.spring_layout(G_dismod_1, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.01)
        pos_dis_1_rescaled = {node: (x*0.1+0.6, y*0.1+0.4, z*0.1+0.7) for node, (x,y,z) in pos_dis_1.items()}
        pos_diseases_1[i] = pos_dis_1_rescaled[i]
    elif i in city_nodes: 
        pos_diseases_1[i] = G.nodes[i]['pos']
    else:
        pos_diseases_1[i] = G.nodes[i]["attrlist"]["init_pos"]

In [75]:
for k,v in pos_diseases_1.items():
    if np.isnan(v[0]) or np.isnan(v[1]) or np.isnan(v[2]):
        pos_diseases_1[k] = (0, 0, 0)
        print("node:", G.nodes[k]['name'])

In [76]:
# node colors 

col_dismod_1_rgba = (77,243,0,200)   
col_dismod_2_rgba = (0,243,143,200)
col_dismod_3_rgba = (184,20,0,200)

nodecol_diseases_1 = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
         nodecol_diseases_1[i] = col_dismod_1_rgba
    elif i in endrin_nodes:
        nodecol_diseases_1[i] = nodecol_endrin[i]
    #elif i in ppi_nodes:
    #    nodecol_diseases_1[i] = d_nodecolors_cmap[i]
    elif i in city_nodes:
        nodecol_diseases_1[i] = (20,20,20,20) #G.nodes[i]['nodecolor']
    else:
        nodecol_diseases_1[i] = (0,0,0,0)

In [77]:
linkcol_diseases_1 = {}
for i in G.edges():
    if i in G_dismod_1.edges():
        linkcol_diseases_1[i] = col_dismod_1_rgba
    elif i[0] in endrin_nodes and i[1] in endrin_nodes:
        linkcol_diseases_1[i] = col_endrin_rgba
    else:
        linkcol_diseases_1[i] = (0,0,0,0)

In [60]:
layout_name = "scene06-Endrin-diseases"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_diseases_1, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_diseases_1, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_diseases_1, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 7 - Darmkrebs und Brustkrebs

In [99]:
# node positions 

pos_diseases_2 = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
        pos_diseases_2[i] = pos_diseases_1[i]
    elif i in d_id_dismod_2.keys():
        pos_dis_2= nx.spring_layout(G_dismod_2, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.5)
        pos_dis_2_rescaled = {node: (x*0.1+0.4, y*0.1+0.2, z*0.1+0.4) for node, (x,y,z) in pos_dis_2.items()}
        pos_diseases_2[i] = pos_dis_2_rescaled[i]
    elif i in d_id_dismod_3.keys():
        pos_dis_3 = nx.spring_layout(G_dismod_3, dim=3, center = (0.5,0.5,0.5), iterations=100, k=0.01)
        pos_dis_3_rescaled = {node: (x*0.1+0.6, y*0.1+0.6, z*0.1+0.4) for node, (x,y,z) in pos_dis_3.items()}
        pos_diseases_2[i] = pos_dis_3_rescaled[i]
    elif i in endrin_nodes:
        pos_diseases_2[i] = G.nodes[i]["attrlist"]["init_pos"]
    elif i in city_nodes: 
        pos_diseases_2[i] = G.nodes[i]['pos']
    else:
        pos_diseases_2[i] = (0.5,0.5,0.5) #G.nodes[i]['pos']

In [100]:
for k,v in pos_diseases_2.items():
    if np.isnan(v[0]) or np.isnan(v[1]) or np.isnan(v[2]):
        pos_diseases_2[k] = (0, 0, 0)
        print("node:", G.nodes[k]['name'])

In [101]:
# node colors 

nodecol_diseases_2 = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
         nodecol_diseases_2[i] = col_dismod_1_rgba
    elif i in d_id_dismod_2.keys():
         nodecol_diseases_2[i] = col_dismod_2_rgba
    elif i in d_id_dismod_3.keys():
        nodecol_diseases_2[i] = col_dismod_3_rgba
    elif i in endrin_nodes:
        nodecol_diseases_2[i] = nodecol_endrin[i]
    #elif i in ppi_nodes:
    #    nodecol_diseases_2[i] = d_nodecolors_cmap[i]
    elif i in city_nodes:
        nodecol_diseases_2[i] = (20,20,20,20) #G.nodes[i]['nodecolor']
    else:
        nodecol_diseases_2[i] = (0,0,0,0)

In [102]:
linkcol_diseases_2 = {}
for i in G.edges():
    if i in G_dismod_1.edges():
        linkcol_diseases_2[i] = col_dismod_1_rgba
    elif i in G_dismod_2.edges():
        linkcol_diseases_2[i] = col_dismod_2_rgba
    elif i[0] in endrin_nodes and i[1] in endrin_nodes:
        linkcol_diseases_2[i] = col_endrin_rgba
    elif i in G_dismod_3.edges():
        linkcol_diseases_2[i] = col_dismod_3_rgba
    elif i in endrin_nodes:
        linkcol_diseases_2[i] = col_endrin_rgba
    #elif i[0] in ppi_nodes and i[1] in ppi_nodes:
    #    linkcol_diseases_2[i] = (0,122,192,100)
    else:
        linkcol_diseases_2[i] = (0,0,0,0)

In [ ]:
import nx2json as nx2j 

nx.set_node_attributes(G_dismod_1, pos_diseases_1, 'pos')
nx.set_node_attributes(G_dismod_1, nodecol_diseases_1, 'nodecolor')
nx.set_edge_attributes(G_dismod_1, linkcol_diseases_1, 'linkcolor')

nx.set_node_attributes(G_dismod_2, pos_diseases_2, 'pos')
nx.set_node_attributes(G_dismod_2, nodecol_diseases_2, 'nodecolor')
nx.set_edge_attributes(G_dismod_2, linkcol_diseases_2, 'linkcolor')

nx.set_node_attributes(G_dismod_3, pos_diseases_2, 'pos')
nx.set_node_attributes(G_dismod_3, nodecol_diseases_2, 'nodecolor')
nx.set_edge_attributes(G_dismod_3, linkcol_diseases_2, 'linkcolor')


# save the subgraphs as json files
G_dismod_1.graph['projectname'] = "Endrin_disease_1"
nx2j.make_json(G_dismod_1, save_json=True)

G_dismod_2.graph['projectname'] = "Endrin_disease_2"
nx2j.make_json(G_dismod_2, save_json=True)

G_dismod_3.graph['projectname'] = "Endrin_disease_3"
nx2j.make_json(G_dismod_3, save_json=True)

✅ Connected to /main
Merged JSON file saved as:  c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\Endrin_disease_1.json
Merged JSON file saved as:  c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\Endrin_disease_2.json
Merged JSON file saved as:  c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\Endrin_disease_3.json


{'directed': False,
 'multigraph': False,
 'projectname': 'Endrin_disease_3',
 'info': 'No description specified.',
 'graphlayouts': ['layoutname_0'],
 'annotationTypes': True,
 'nodes': [{'id': 0, 'name': 'MLH1', 'annotation': {}},
  {'id': 1, 'name': 'COX2', 'annotation': {}},
  {'id': 2, 'name': 'SMAD4', 'annotation': {}},
  {'id': 3, 'name': 'HAPLN1', 'annotation': {}},
  {'id': 4, 'name': 'APC', 'annotation': {}},
  {'id': 5, 'name': 'TCF7L2', 'annotation': {}},
  {'id': 6, 'name': 'DLC1', 'annotation': {}},
  {'id': 7, 'name': 'IGFBP3', 'annotation': {}},
  {'id': 8, 'name': 'SMAD7', 'annotation': {}},
  {'id': 9, 'name': 'HHIP', 'annotation': {}},
  {'id': 10, 'name': 'CUBN', 'annotation': {}},
  {'id': 11, 'name': 'BMP4', 'annotation': {}},
  {'id': 12, 'name': 'SYNE1', 'annotation': {}},
  {'id': 13, 'name': 'PRKCE', 'annotation': {}},
  {'id': 14, 'name': 'SMAD3', 'annotation': {}},
  {'id': 15, 'name': 'FGFR3', 'annotation': {}},
  {'id': 16, 'name': 'EVL', 'annotation': {}}

✅ Connected to /main
✅ Connected to /main


In [104]:
layout_name = "scene07-Endrin-diseases"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_diseases_2, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_diseases_2, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_diseases_2, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


### SCENE 8 - TRANSITION SCENE 

In [91]:
# node colors 

nodecol_transition = {}
for i in G.nodes():
    if i in d_id_dismod_1.keys():
         nodecol_transition[i] = col_dismod_1_rgba
    elif i in d_id_dismod_2.keys():
         nodecol_transition[i] = col_dismod_2_rgba
    elif i in d_id_dismod_3.keys():
        nodecol_transition[i] = col_dismod_3_rgba
    elif i in city_nodes:
        nodecol_transition[i] = (20,20,20,20) #G.nodes[i]['nodecolor']
    else:
        nodecol_transition[i] = (0,0,0,0)

In [92]:
linkcol_transition = {}

layout_name = "scene08-transition"
new_tex_nodes = tex_gen.generate_node_color_texture(nodecol_transition, layout_name, save=True)
new_tex_links = tex_gen.generate_link_color_texture(linkcol_transition, layout_name, save=True)
new_tex_nodepos = tex_gen.generate_node_position_texture(pos_diseases_2, layout_name, save=True, normalize_flag=False)

# Emit to server
nodeRGB_path_rel = f"static/projects/{session.sel_name}/layoutsRGB/{layout_name}.png"
linksRGB_path_rel = f"static/projects/{session.sel_name}/linksRGB/{layout_name}.png"
nodeXYZ_path_rel_high = f"static/projects/{session.sel_name}/layouts/{layout_name}.png"
nodeXYZ_path_rel_low = f"static/projects/{session.sel_name}/layoutsl/{layout_name}l.png"

session.client.emit("ex", {
    "usr": session.client.uid,
    "id":None,
    "fn": "updateTempTex",
    "textures": [
        {"channel": "nodeRGB", "path": nodeRGB_path_rel},
        {"channel": "linkRGB", "path": linksRGB_path_rel},
        {"channel": "nodeXYZ", "path": nodeXYZ_path_rel_high},
        {"channel": "nodeXYZl", "path": nodeXYZ_path_rel_low}
    ]
}, namespace=session.client.namespace)


✅ Connected to /main


# snippets

In [ ]:
import pandas as pd

from math import radians, cos, sin, sqrt, atan, atan2
def geodetic_to_geocentric(ellipsoid, lat, lon, height):
    # Adjust the flattening factor to influence the ellipsoid shape
    a, rf = ellipsoid
    # rf *= 1.2  # Increase the flattening factor to bring poles closer together

    phi = radians(lat)
    lamb = radians(lon)

    sin_phi = sin(phi)
    e2 = 1 - (1 - 1 / rf) ** 2
    n = a / sqrt(1 - e2 * sin_phi ** 2) 
    
    r = ((n + height) * cos(phi))
    x = r * cos(lamb)
    y = r * sin(lamb)   
    z = (n * (1 - e2) + height) * sin(phi) 

    return x, y, z  


def geocentric_to_geodetic(ellipsoid, x, y, z):
    a, rf = ellipsoid
    e2 = 1 - (1 - 1 / rf) ** 2
    b = a * (1 - 1 / rf)  # Semi-minor axis
    p = sqrt(x**2 + y**2)
    theta = atan(z * a / (p * b))
    sin_theta = sin(theta)
    cos_theta = cos(theta)
    
    lat = atan((z + e2 * b * sin_theta**3) / (p - e2 * a * cos_theta**3))
    lon = atan2(y, x)
    
    n = a / sqrt(1 - e2 * sin(lat)**2)
    h = p / cos(lat) - n
    
    return lat, lon, h



def normalize_coordinates(x, y, z):
    magnitude = sqrt(x**2 + y**2 + z**2)
    x_norm, y_norm, z_norm = x / magnitude, y / magnitude, z / magnitude
    x_final = (x_norm+1)/2
    y_final = (y_norm+1)/2
    z_final = (z_norm+1)/2
    return x_final, y_final, z_final


df = pd.read_csv("temp-files/Pesticides/world-data-2023.csv")
pos_latlon = df[["Latitude", "Longitude"]].values.tolist()
d_pos_latlon = dict(zip(df["Country"], pos_latlon))

pos_xyz = {}

flattening_factor = 298.257223563
WGS84_radius = 6378137.0

for k,v in d_pos_latlon.items():
    x,y,z = geodetic_to_geocentric((WGS84_radius, flattening_factor), v[0], v[1], 0)
    xn, yn, zn = normalize_coordinates(x,y,z)
    pos_xyz[k] = (y,x,z*0.8) #(yn,xn,zn)